# Análisis modular del TP1

Este notebook ejecuta el flujo de análisis por celdas y es compatible con VS Code, Jupyter y Google Colab.

**Archivo de entrada:** `datos_TP1_2025.csv`

## 1. Crear el notebook y seleccionar el kernel

En VS Code o Colab, seleccioná un kernel de Python válido antes de ejecutar las celdas. Los resultados quedarán embebidos en este archivo `.ipynb`.

## 2. Instalar y validar dependencias

Ejecutá esta celda solo si el entorno no tiene `pandas` o `matplotlib` instalados. En Google Colab normalmente ya están disponibles.

In [24]:
%pip install pandas matplotlib

You should consider upgrading via the '/Users/diegorojas/Documents/github projects/aa-tp-1/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [25]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.append(str(Path.cwd().parent))
from functions.utils import *

## 3. Cargar los datos

La ruta relativa funciona tanto en la carpeta local del proyecto como después de subir el archivo a Colab.

In [26]:
from pathlib import Path

DATA_PATH = Path("../dataset/dataset_TP1_2026.csv")
OUTPUT_DIR = Path("../outputs")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró: {DATA_PATH.resolve()}")

dataset = pd.read_csv(DATA_PATH)
print(f"Dataset cargado: {dataset.shape[0]:,} filas y {dataset.shape[1]} columnas")
dataset.head()

Dataset cargado: 50,000 filas y 28 columnas


,Estudios_máximos_antes_de_la_inscripción,estado_civil,modo_aplicacion,sexo,desplazado,target,Carrera,Asistencia_diurna/nocturna,Cualificación_promedio_de_estudios_previos,Puntaje_en_examen_de_ingreso,...,Unidades_curriculares_1er_semestre_(aprobadas),Nota_promedio_en_el_1er_semestre,Unidades_curriculares_2do_semestre_(inscrito),Cantidad_de_evaluaciones_en_el_2do_semestre_,Unidades_curriculares_2do_semestre_(aprobadas),Nota_promedio_en_el_2do_semestre,Demanda_Cog_Padre,Demanda_Cog_Madre,Ingresos_Padre,Ingresos_Madre
0,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,Graduado,Turismo,Diurna,65,61,...,6,63,6,8,5,69,Muy baja demanda cognitiva,Muy baja demanda cognitiva,Bajos ingresos,Bajos ingresos
1,Educación Secundaria,Soltero,Acceso por Normativa Específica,Femenino,No,Desertor,Enfermería,Diurna,67,58,...,0,0,7,7,0,0,Muy baja demanda cognitiva,Muy baja demanda cognitiva,Bajos ingresos,Bajos ingresos
2,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Gestión,Diurna,64,62,...,3,53,5,10,2,52,Muy baja demanda cognitiva,Demanda cognitiva media,Bajos ingresos,Ingresos medios
3,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Enfermería,Diurna,67,54,...,5,63,7,8,3,58,Demanda cognitiva media,Demanda cognitiva media,Ingresos medios,Ingresos medios
4,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Publicidad y Marketing,Diurna,68,62,...,6,62,6,9,6,65,Muy alta demanda cognitiva,Muy baja demanda cognitiva,Altos ingresos,Bajos ingresos


In [27]:
dataset.columns

Index(['Estudios_máximos_antes_de_la_inscripción', 'estado_civil',
       'modo_aplicacion', 'sexo', 'desplazado', 'target', 'Carrera',
       'Asistencia_diurna/nocturna',
       'Cualificación_promedio_de_estudios_previos',
       'Puntaje_en_examen_de_ingreso', 'Necesidades_educativas_especiales',
       'Tiene_deuda', 'Pago_de_matrícula_al_día', 'Poseedor_de_Beca',
       'Edad_al_momento_de_la_inscripción', 'Estudiante_Internacional',
       'Unidades_curriculares_1er_semestre_(inscrito)',
       'Cantidad_de_evaluaciones_en_el_1er_semestre',
       'Unidades_curriculares_1er_semestre_(aprobadas)',
       'Nota_promedio_en_el_1er_semestre',
       'Unidades_curriculares_2do_semestre_(inscrito)',
       'Cantidad_de_evaluaciones_en_el_2do_semestre_',
       'Unidades_curriculares_2do_semestre_(aprobadas)',
       'Nota_promedio_en_el_2do_semestre', 'Demanda_Cog_Padre',
       'Demanda_Cog_Madre', 'Ingresos_Padre', 'Ingresos_Madre'],
      dtype='object')

In [28]:
columns_mapping = {
    'Estudios_máximos_antes_de_la_inscripción_label': 'estudios_maximos', 
    'estado_civil': 'estado_civil',
    'modo_aplicacion': 'modo_aplicacion', 
    'sexo': 'sexo', 
    'desplazado': 'desplazado',
    'target': 'target', 
    'Carrera': 'carrera', 
    'Asistencia_diurna/nocturna': 'asistencia', 
    'Cualificación_promedio_de_estudios_previos': 'cualificacion_promedio', 
    'Puntaje_en_examen_de_ingreso': 'puntaje_ingreso', 
    'Necesidades_educativas_especiales': 'necesidades_educativas_especiales',
    'Tiene_deuda': 'tiene_deuda', 
    'Pago_de_matrícula_al_día': 'matricula_al_dia', 
    'Poseedor_de_Beca': 'posee_beca',
    'Edad_al_momento_de_la_inscripción': 'edad_inscripcion', 
    'Estudiante_Internacional': 'estudiante_internacional',
    'Unidades_curriculares_1er_semestre_(inscrito)': 'uc_inscritos_primer_semestre',
    'Cantidad_de_evaluaciones_en_el_1er_semestre': 'cant_eval_primer_semestre',
    'Unidades_curriculares_1er_semestre_(aprobadas)': 'uc_aprobadas_primer_semestre',
    'Nota_promedio_en_el_1er_semestre': 'nota_promedio_primer_semestre',
    'Unidades_curriculares_2do_semestre_(inscrito)': 'uc_inscritos_segundo_semestre',
    'Cantidad_de_evaluaciones_en_el_2do_semestre_': 'cant_eval_segundo_semestre',
    'Unidades_curriculares_2do_semestre_(aprobadas)': 'uc_aprobadas_segundo_semestre',
    'Nota_promedio_en_el_2do_semestre': 'nota_promedio_segundo_semestre', 
    'Demanda_Cog_Padre_label': 'demanda_cog_padre',
    'Demanda_Cog_Madre_label': 'demanda_cog_madre', 
    'Ingresos_Padre_label': 'ingresos_padre',
    'Ingresos_Madre_label': 'ingresos_madre'  
}

dataset_renamed = dataset.rename(columns=columns_mapping)

In [29]:
dataset_renamed.head()

,Estudios_máximos_antes_de_la_inscripción,estado_civil,modo_aplicacion,sexo,desplazado,target,carrera,asistencia,cualificacion_promedio,puntaje_ingreso,...,uc_aprobadas_primer_semestre,nota_promedio_primer_semestre,uc_inscritos_segundo_semestre,cant_eval_segundo_semestre,uc_aprobadas_segundo_semestre,nota_promedio_segundo_semestre,Demanda_Cog_Padre,Demanda_Cog_Madre,Ingresos_Padre,Ingresos_Madre
0,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,Graduado,Turismo,Diurna,65,61,...,6,63,6,8,5,69,Muy baja demanda cognitiva,Muy baja demanda cognitiva,Bajos ingresos,Bajos ingresos
1,Educación Secundaria,Soltero,Acceso por Normativa Específica,Femenino,No,Desertor,Enfermería,Diurna,67,58,...,0,0,7,7,0,0,Muy baja demanda cognitiva,Muy baja demanda cognitiva,Bajos ingresos,Bajos ingresos
2,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Gestión,Diurna,64,62,...,3,53,5,10,2,52,Muy baja demanda cognitiva,Demanda cognitiva media,Bajos ingresos,Ingresos medios
3,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Enfermería,Diurna,67,54,...,5,63,7,8,3,58,Demanda cognitiva media,Demanda cognitiva media,Ingresos medios,Ingresos medios
4,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Publicidad y Marketing,Diurna,68,62,...,6,62,6,9,6,65,Muy alta demanda cognitiva,Muy baja demanda cognitiva,Altos ingresos,Bajos ingresos


In [ ]:
macro_categoria_carrera.update({
    "Servicio Social": "Ciencias sociales y educación",
    "Educación Básica": "Ciencias sociales y educación",
})

dataset_renamed["macro_categoria_carrera"] = dataset_renamed["carrera"].map(
    macro_categoria_carrera
)

dataset_renamed["macro_categoria_carrera"].value_counts()


Carreras sin categoría: []


macro_categoria_carrera
Comunicación, marketing y diseño         12490
Administración y turismo                 11254
Salud                                     8377
Ciencias agropecuarias y veterinarias     7002
Ciencias sociales                         6778
Educación                                 2090
Ingeniería y tecnología                   2009
Name: count, dtype: int64

In [30]:
# Transformar los campos a numericos

dataset_renamed['es_masculino'] = np.where(dataset_renamed['sexo'] == 'Femenino', 0, 1)
dataset_renamed['es_desplazado'] = np.where(dataset_renamed['desplazado'] == 'No', 0, 1)
dataset_renamed['es_asistencia_diurna'] = np.where(dataset_renamed['asistencia'] == 'Diurna', 1, 0)
dataset_renamed['es_deudor'] = np.where(dataset_renamed['tiene_deuda'] == 'Sí', 1, 0)
dataset_renamed['es_moroso'] = np.where(dataset_renamed['matricula_al_dia'] == 'Sí', 1, 0)
dataset_renamed['es_becado'] = np.where(dataset_renamed['posee_beca'] == 'Sí', 1, 0)
dataset_renamed['es_estudiante_internacional'] = np.where(dataset_renamed['estudiante_internacional'] == 'Sí', 1, 0)
dataset_renamed['tiene_necesidades_educativas_especiales'] = np.where(dataset_renamed['necesidades_educativas_especiales'] == 'Sí', 1, 0)

In [31]:
# Promedios entre el primer y segundo semestre para cada estudiante
dataset_renamed["promedio_notas_semestres"] = dataset_renamed[
    [
        "nota_promedio_primer_semestre",
        "nota_promedio_segundo_semestre",
    ]
].mean(axis=1)

dataset_renamed["promedio_uc_inscritos_semestres"] = dataset_renamed[
    [
        "uc_inscritos_primer_semestre",
        "uc_inscritos_segundo_semestre",
    ]
].mean(axis=1)

dataset_renamed["promedio_uc_aprobadas_semestres"] = dataset_renamed[
    [
        "uc_aprobadas_primer_semestre",
        "uc_aprobadas_segundo_semestre",
    ]
].mean(axis=1)

dataset_renamed["promedio_evaluaciones_semestres"] = dataset_renamed[
    [
        "cant_eval_primer_semestre",
        "cant_eval_segundo_semestre",
    ]
].mean(axis=1)

convirtiendo las variables ingresos a numericas (categoricas ordinales)

In [32]:
ingresos_mapping = {
    "Bajos ingresos": 1,
    "Ingresos medios-bajos": 2,
    "Ingresos medios": 3,
    "Ingresos medios-altos": 4,
    "Altos ingresos": 5,
}

dataset_renamed["ingresos_padre_nivel"] = (
    dataset_renamed["Ingresos_Padre"].map(ingresos_mapping)
)

dataset_renamed["ingresos_madre_nivel"] = (
    dataset_renamed["Ingresos_Madre"].map(ingresos_mapping)
)


In [33]:
dataset_renamed["ingresos_familia_nivel"] = dataset_renamed[
    ["ingresos_padre_nivel", "ingresos_madre_nivel"]
].mean(axis=1)

convirtiendo las variables cognitivas a numericas (categoricas ordinales)

In [34]:
demanda_cog_mapping = {
    "Muy baja demanda cognitiva": 1,
    "Demanda cognitiva baja": 2,
    "Demanda cognitiva media": 3,
    "Alta demanda cognitiva": 4,
    "Muy alta demanda cognitiva": 5,
}

dataset_renamed["demanda_cog_padre_nivel"] = (
    dataset_renamed["Demanda_Cog_Padre"].map(demanda_cog_mapping)
)

dataset_renamed["demanda_cog_madre_nivel"] = (
    dataset_renamed["Demanda_Cog_Madre"].map(demanda_cog_mapping)
)

In [35]:
dataset_renamed["demanda_cog_familia_nivel"] = dataset_renamed[
    ["demanda_cog_padre_nivel", "demanda_cog_madre_nivel"]
].mean(axis=1)

In [36]:
dataset_renamed.head()

,Estudios_máximos_antes_de_la_inscripción,estado_civil,modo_aplicacion,sexo,desplazado,target,carrera,asistencia,cualificacion_promedio,puntaje_ingreso,...,promedio_notas_semestres,promedio_uc_inscritos_semestres,promedio_uc_aprobadas_semestres,promedio_evaluaciones_semestres,ingresos_padre_nivel,ingresos_madre_nivel,ingresos_familia_nivel,demanda_cog_padre_nivel,demanda_cog_madre_nivel,demanda_cog_familia_nivel
0,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,Graduado,Turismo,Diurna,65,61,...,66.0,6.0,5.5,7.0,1,1,1.0,1,1,1.0
1,Educación Secundaria,Soltero,Acceso por Normativa Específica,Femenino,No,Desertor,Enfermería,Diurna,67,58,...,0.0,7.0,0.0,12.5,1,1,1.0,1,1,1.0
2,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Gestión,Diurna,64,62,...,52.5,5.0,2.5,8.5,1,3,2.0,1,3,2.0
3,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Enfermería,Diurna,67,54,...,60.5,7.0,4.0,8.5,3,3,3.0,3,3,3.0
4,Educación Secundaria,Soltero,Acceso General,Femenino,Sí,En Curso,Publicidad y Marketing,Diurna,68,62,...,63.5,6.0,6.0,7.5,5,1,3.0,5,1,3.0


In [37]:
guardar_dataframe_pickle(dataset_renamed, "dataset_cleaned", OUTPUT_DIR)

DataFrame guardado en: /Users/diegorojas/Documents/github projects/aa-tp-1/outputs/dataset_cleaned.pkl


PosixPath('../outputs/dataset_cleaned.pkl')